In [8]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
df = pd.read_csv("C:/Users/Martin/Desktop/Github/eu-enlargement-brain-gain-did/Data/student_data_2000_3_6_9.csv")

C:\Users\Martin\AppData\Local\Temp\ipykernel_93448\2555057915.py:5: DtypeWarning: Columns (4,5,7,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/Martin/Desktop/Github/eu-enlargement-brain-gain-did/Data/student_data_2000_3_6_9.csv")


In [9]:
test_countries = ["POL","CZE","HUN","LVA"]
main_controls = ["BEL", "FRA", "GRC", "IRL", "ISL", "ITA", "LUX", "NLD", "NOR", 
                     "PRT","SCW", "CHE", "ESP","DNK", "DEU", "FIN","RUS", "AUT", "SWE"]
neighbour_controls = ["DNK", "DEU", "FIN", "RUS","AUT","SWE"]

def prepare_data(control_countries):

    df1 = df.copy()

    def land(x):
        if x in test_countries:
            return "test"
        elif x in control_countries:
            return "control"
        else:
            return "nothing"

   
    df1["group"] = df1["country"].apply(land)
    df1 = df1[df1["group"] != "nothing"].copy()
    df1["treatment"] = np.where(df1["group"] == "test",1,0)
    df1["post"] = np.where(df1["year"].isin([2006,2009]), 1,0)
    df1["did"] = df1["treatment"] * df1["post"]
    df1["year"] = df1["year"].astype(str)

    return df1
df_main = prepare_data(main_controls)

df_neighbour = prepare_data(neighbour_controls)

In [10]:
outcomes = ["math", "read", "science"]
controls = [ "gender","internet","desk","room","dishwasher","television","computer_n","car","book","wealth","escs"]
results_main = {}

for outcome in outcomes:

    data = df_main.dropna(subset=[outcome] + controls).copy()

    formula = (f"{outcome} ~ did " "+ gender ""+ internet ""+ desk ""+ room ""+ dishwasher ""+ television ""+ computer_n ""+ car ""+ book ""+ wealth ""+ escs ""+ C(country) ""+ C(year)")

    model = smf.ols(
        formula,
        data=data
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": data["country"]}
    )

    results_main[outcome] = model

In [11]:
table_main = summary_col(
    [ results_main["math"], results_main["read"],results_main["science"]],
    stars=True,
    model_names=["Math", "Reading","Science"],
    regressor_order=["did","gender[T.male]","wealth","escs"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R²": lambda x: f"{x.rsquared:.3f}",
        "Country FE": lambda x: "Yes",
        "Year FE": lambda x: "Yes"}
)
print(table_main)


                          Math      Reading     Science  
---------------------------------------------------------
did                   18.2869***  25.9166***  16.4312**  
                      (4.9223)    (6.0466)    (6.9990)   
gender[T.male]        15.7914***  -34.7608*** 5.5530***  
                      (1.1925)    (1.6634)    (1.1649)   
wealth                -18.6486*** -19.9316*** -20.9901***
                      (2.2685)    (1.9698)    (2.1630)   
escs                  20.9909***  23.7197***  22.8934*** 
                      (2.0259)    (1.7563)    (1.7693)   
Intercept             381.9377*** 399.5044*** 385.8150***
                      (8.8978)    (9.6615)    (10.1419)  
internet[T.yes]       18.1036***  19.1938***  17.1640*** 
                      (1.1879)    (1.5161)    (1.7269)   
desk[T.yes]           9.7382***   11.1484***  8.4893***  
                      (1.9281)    (1.8972)    (1.9565)   
room[T.yes]           8.6412***   7.9575***   10.1878*** 
             

In [12]:
outcomes = ["math", "read", "science"]
controls = [ "gender","internet","desk","room","dishwasher","television","computer_n","car","book","wealth","escs"]
results_neighbour = {}

for outcome in outcomes:

    data = df_neighbour.dropna(subset=[outcome] + controls).copy()

    formula = (f"{outcome} ~ did " "+ gender ""+ internet ""+ desk ""+ room ""+ dishwasher ""+ television ""+ computer_n ""+ car ""+ book ""+ wealth ""+ escs ""+ C(country) ""+ C(year)")

    model = smf.ols(
        formula,
        data=data
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": data["country"]}
    )

    results_neighbour[outcome] = model

In [13]:
table_neighbour = summary_col(
    [
        results_neighbour["math"],
        results_neighbour["read"],
        results_neighbour["science"]
    ],
    stars=True,
    model_names=[
        "Math",
        "Reading",
        "Science"
    ],
    regressor_order=[
        "did",
        "gender[T.male]",
        "wealth",
        "escs"
    ],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "R²": lambda x: f"{x.rsquared:.3f}",
        "Country FE": lambda x: "Yes",
        "Year FE": lambda x: "Yes"
    }
)

print(table_neighbour)


                          Math      Reading     Science  
---------------------------------------------------------
did                   20.7362***  26.0025***  13.7572*   
                      (5.7865)    (6.9472)    (7.9542)   
gender[T.male]        13.4777***  -37.3371*** 4.9021***  
                      (2.0543)    (1.5221)    (1.8121)   
wealth                -15.9471*** -16.8642*** -16.9606***
                      (4.8353)    (4.0030)    (4.2955)   
escs                  26.4569***  28.2582***  27.8180*** 
                      (2.4346)    (1.8858)    (1.8856)   
Intercept             399.7847*** 415.1770*** 403.0737***
                      (19.4824)   (18.7309)   (20.5319)  
internet[T.yes]       17.0044***  17.2175***  14.4201*** 
                      (2.9490)    (2.2387)    (2.7121)   
desk[T.yes]           6.8470***   7.3945***   5.3288***  
                      (1.6501)    (2.0153)    (1.9083)   
room[T.yes]           3.5312      2.9939      3.1687     
             